In [ ]:
# hide
import time

import numpy as np
import pyquist as pq

In [ ]:
# Offline synthesis: x[n] = f(n, c[n]), with the control c[n] (pitch) known ahead of time for all n
f_s = 44100
melody = [60, 64, 67, 72, 67, 64, 60]
c = np.repeat(melody, f_s // 4).astype(np.float64)
f = pq.helper.pitch_to_frequency(c)
x = np.sin(np.cumsum(2.0 * np.pi * f / f_s))
pq.play(pq.Audio(x, f_s))

In [ ]:
# Real-time synthesis: the control c_k = c[kB] is read once per block, and the
# phase is state, carried from one block to the next
class Sinusoid:
    def __init__(self, pitch: float = 0.0) -> None:
        self.pitch, self.phase = (pitch, 0.0)

    def callback(self, block: pq.Audio) -> None:
        f = pq.helper.pitch_to_frequency(self.pitch)  # read c_k
        B = len(block)
        dphase = 2.0 * np.pi * f / block.sample_rate
        phases = self.phase + dphase * np.arange(B)
        block[:] = np.sin(phases)[:, np.newaxis]
        self.phase = (self.phase + dphase * B) % (2.0 * np.pi)


# Simulate the audio system, which asks for the next block every BΔt seconds,
# and a user who switches to the next note of the melody right on time.
# Try B = 1, and compare how much of the time budget it uses.
# Try B = 16384: the notes arrive late, and some are skipped entirely!
B = 512
synth = Sinusoid(melody[0])
blocks = [pq.Audio.zeros(B, 1, f_s)]       # x_0: silence, queued before playback starts
start = time.perf_counter()
for k in range(len(c) // B):
    synth.pitch = c[k * B]                 # the user's pitch at time kBΔt
    block = pq.Audio.zeros(B, 1, f_s)
    synth.callback(block)                  # synthesize x_{k+1}
    blocks.append(block)
busy = time.perf_counter() - start         # seconds spent synthesizing
print(f"B = {B}: {1000 * B / f_s:.2f} ms latency, {100 * busy / (len(c) / f_s):.1f}% of time budget")
pq.play(pq.Audio.concatenate(blocks))